# 05. ReAct Agent

## What you'll learn

- What the **ReAct** (Reason + Act) pattern is and why interleaving reasoning with actions improves agent performance
- How to write a **system prompt** with few-shot examples that teaches the LLM the Thought / Action / Observation format
- How to **parse** ReAct-formatted responses into structured components
- How the **scratchpad** accumulates a full reasoning trace across multiple steps
- How to build a **complete ReAct agent loop** with tool dispatch, error handling, and iteration limits
- How to **trace and debug** agent behavior by inspecting the scratchpad

## Prerequisites

- [01. Hello LLM](01_hello_llm.ipynb) — making API calls with `chat()`
- [02. Basic Agent Loop](02_basic_agent_loop.ipynb) — the prompt-respond-loop pattern
- [03. Tool Use from Scratch](03_tool_use_from_scratch.ipynb) — defining tools and dispatching calls
- [04. Structured Output](04_structured_output.ipynb) — parsing structured LLM responses
- [Appendix 04. Strings and JSON](../appendix/04_strings_and_json.ipynb) — regex, JSON parsing, f-strings

## Why this matters

In notebook 03 we built an agent that can use tools, but it jumps straight from the user's question to a tool call. There's no visible *reasoning* step — the model picks a tool and hopes for the best. The **ReAct** pattern (Yao et al., 2022) fixes this by requiring the model to produce an explicit **Thought** before every **Action**. This simple change has a dramatic effect: the agent plans better, recovers from mistakes more gracefully, and produces a human-readable trace you can use for debugging.

This notebook is the **culmination of Phase 1**. Every concept from notebooks 01 through 04 — API calls, agent loops, tool use, structured output parsing — comes together here in a single, complete agent.

> **Further reading:**
> - [ReAct: Synergizing Reasoning and Acting in Language Models (Yao et al., 2022)](https://arxiv.org/abs/2210.03629) — the original paper
> - [LLM Powered Autonomous Agents (Lilian Weng)](https://lilianweng.github.io/posts/2023-06-23-agent/) — excellent overview of agent architectures including ReAct

---

## 1. What is ReAct?

The core insight of ReAct is simple: **interleaving reasoning with acting** produces better agents than either reasoning or acting alone.

In notebook 03, our tool-use agent went directly from the user's question to a tool call:

```
User: What's the weather in Paris?
Action: get_weather
Action Input: {"city": "Paris"}
```

ReAct adds an explicit **Thought** step before each action. The model must articulate *why* it's choosing a particular tool and *what* it expects to learn:

```
User: If it's above 20C in Paris, calculate a 15% tip on $45.

Thought: I need to find the weather in Paris first to check the temperature.
Action: get_weather
Action Input: {"city": "Paris"}
Observation: 22C, sunny

Thought: It's 22C, which is above 20C. So I need to calculate a 15% tip on $45.
Action: calculate
Action Input: {"expression": "45 * 0.15"}
Observation: 6.75

Thought: I have all the information needed.
FINAL ANSWER: The weather in Paris is 22C (sunny), which is above 20C, so the tip is 15%. A 15% tip on a $45 dinner is $6.75.
```

Notice the pattern:

| Component | Source | Purpose |
|-----------|--------|---------|
| **Thought** | LLM generates | Reasoning — plan, reflect, decide |
| **Action** | LLM generates | Which tool to call |
| **Action Input** | LLM generates | Arguments for the tool (JSON) |
| **Observation** | Our code injects | Result from executing the tool |

The **Observation** is the key difference from pure chain-of-thought: it's *real data from the environment*, not something the LLM imagined. Our code executes the tool and injects the result back into the conversation.

---

## 2. The ReAct Format

Let's define each component precisely:

**Thought** — The model's internal reasoning. It should explain:
- What information it has so far
- What information it still needs
- Why it's choosing a particular action

**Action** — The name of the tool to call. Must exactly match one of the available tools.

**Action Input** — The arguments to pass to the tool, formatted as JSON.

**Observation** — The result from executing the tool. This is **not generated by the LLM** — our agent code runs the tool and appends the observation to the conversation. This grounds the agent in reality.

**FINAL ANSWER** — When the agent has enough information, it stops calling tools and returns a complete answer. This is the termination signal for our loop.

The loop looks like this:

```
repeat:
    LLM generates -> Thought + (Action + Action Input  OR  FINAL ANSWER)
    if FINAL ANSWER: return answer
    else: execute tool, get Observation, append to conversation
```

---

## 3. Setup

First, let's import `chat` from our shared utilities and the libraries we'll need.

In [ ]:
import sys
sys.path.insert(0, "..")
from utils import chat

In [ ]:
import re
import json
import time

---

## 4. Building the ReAct Prompt

The system prompt is where we teach the LLM the ReAct format. It needs three things:

1. **Format instructions** — exactly how to structure Thought/Action/Action Input and FINAL ANSWER
2. **Tool descriptions** — what tools are available and how to call them
3. **A few-shot example** — a concrete demonstration of the format in action

The few-shot example is crucial. Without it, models often drift from the format or forget to include the Thought step.

In [ ]:
REACT_SYSTEM_PROMPT = """You are a helpful assistant that solves problems step by step using tools.

You have access to the following tools:
{tool_descriptions}

To use a tool, you MUST respond in EXACTLY this format:

Thought: <your reasoning about what to do next>
Action: <tool_name>
Action Input: <json arguments>

When you have enough information to answer, respond in EXACTLY this format:

Thought: <your reasoning about why you're done>
FINAL ANSWER: <your complete answer>

IMPORTANT: Always start with a Thought. Never skip the Thought step.

Here is an example:

User: What is 25 * 48?

Thought: I need to calculate 25 * 48. Let me use the calculator.
Action: calculate
Action Input: {{"expression": "25 * 48"}}

Observation: 1200

Thought: The calculation is complete.
FINAL ANSWER: 25 * 48 = 1200"""

Notice the `{{` and `}}` in the few-shot example. Since this string will be used with `.format()`, we need double braces to produce literal `{` and `}` characters in the output. This is the same f-string escaping we covered in [Appendix 04](../appendix/04_strings_and_json.ipynb).

Let's verify the prompt renders correctly:

In [ ]:
# Quick check — the double braces should render as single braces
sample = REACT_SYSTEM_PROMPT.format(tool_descriptions="- calculate(expression): Evaluate a math expression.")
print(sample[-200:])

---

## 5. Defining Tools

We'll reuse the same three tools from notebook 03 — `calculate`, `get_weather`, and `search` — so this notebook stays self-contained. Each tool is a plain Python function that takes keyword arguments and returns a string.

In [ ]:
def calculate(expression: str) -> str:
    """Evaluate a math expression. Only allows numbers and basic operators."""
    try:
        allowed = set("0123456789+-*/(). ")
        if not all(c in allowed for c in expression):
            return "Error: expression contains invalid characters"
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"


def get_weather(city: str) -> str:
    """Get current weather for a city (simulated for reproducibility)."""
    import random
    random.seed(hash(city) % 100)  # deterministic per city
    temp = random.randint(15, 35)
    conditions = random.choice(["sunny", "cloudy", "rainy", "partly cloudy"])
    return f"{temp}C, {conditions}"


def search(query: str) -> str:
    """Simulated search tool with a small knowledge base."""
    results = {
        "population of paris": "The population of Paris is approximately 2.1 million in the city proper.",
        "height of eiffel tower": "The Eiffel Tower is 330 meters tall.",
        "speed of light": "The speed of light is approximately 299,792,458 meters per second.",
    }
    query_lower = query.lower()
    for key, value in results.items():
        if key in query_lower:
            return value
    return f"Search results for '{query}': No specific results found. Try a more specific query."

In [ ]:
# Test each tool to make sure they work
print("calculate:", calculate("25 * 48"))
print("get_weather:", get_weather("Paris"))
print("search:", search("height of Eiffel Tower"))

Now we set up the **tool registry** (name-to-function mapping) and the **tool descriptions** string that goes into the system prompt:

In [ ]:
TOOL_REGISTRY = {
    "calculate": calculate,
    "get_weather": get_weather,
    "search": search,
}

TOOL_DESCRIPTIONS = """- calculate(expression): Evaluate a math expression. Example: calculate("2 + 3 * 4")
- get_weather(city): Get current weather for a city. Example: get_weather("Paris")
- search(query): Search for information. Example: search("population of Paris")"""

print(TOOL_DESCRIPTIONS)

---

## 6. Parsing ReAct Responses

When the LLM responds, it will (hopefully) follow our format. We need a parser that extracts:

- The **Thought** (always present)
- Either an **Action** + **Action Input** (tool call) or a **FINAL ANSWER** (termination)

The parser needs to be robust — LLMs don't always follow instructions perfectly. We use regex to extract each component.

In [ ]:
def parse_react_response(response):
    """Parse a ReAct-format response into components.
    
    Returns a dict with keys:
        thought: str or None
        action: str or None
        action_input: dict or None
        final_answer: str or None
    """
    result = {"thought": None, "action": None, "action_input": None, "final_answer": None}
    
    # Extract Thought
    thought_match = re.search(
        r"Thought:\s*(.+?)(?=\nAction:|\nFINAL ANSWER:|\Z)", response, re.DOTALL
    )
    if thought_match:
        result["thought"] = thought_match.group(1).strip()
    
    # Check for FINAL ANSWER
    final_match = re.search(r"FINAL ANSWER:\s*(.+)", response, re.DOTALL)
    if final_match:
        result["final_answer"] = final_match.group(1).strip()
        return result
    
    # Extract Action and Action Input
    action_match = re.search(r"^Action:\s*(.+)$", response, re.MULTILINE)
    input_match = re.search(r"^Action Input:\s*(.+)$", response, re.MULTILINE)
    
    if action_match:
        result["action"] = action_match.group(1).strip()
    if input_match:
        raw_input = input_match.group(1).strip()
        try:
            result["action_input"] = json.loads(raw_input)
        except json.JSONDecodeError:
            result["action_input"] = {"raw": raw_input}
    
    return result

Let's test the parser with different response formats to make sure it handles all the cases:

In [ ]:
# Test 1: Action response
test_action = """Thought: I need to calculate this expression.
Action: calculate
Action Input: {"expression": "25 * 48"}"""

parsed = parse_react_response(test_action)
print("Test 1 — Action response:")
print(f"  Thought: {parsed['thought']}")
print(f"  Action: {parsed['action']}")
print(f"  Action Input: {parsed['action_input']}")
print(f"  Final Answer: {parsed['final_answer']}")

In [ ]:
# Test 2: Final answer response
test_final = """Thought: I have all the information I need.
FINAL ANSWER: The answer is 42."""

parsed = parse_react_response(test_final)
print("Test 2 — Final answer:")
print(f"  Thought: {parsed['thought']}")
print(f"  Action: {parsed['action']}")
print(f"  Final Answer: {parsed['final_answer']}")

In [ ]:
# Test 3: Malformed action input (not valid JSON)
test_bad_json = """Thought: Let me search for this.
Action: search
Action Input: population of Paris"""

parsed = parse_react_response(test_bad_json)
print("Test 3 — Malformed JSON (graceful fallback):")
print(f"  Thought: {parsed['thought']}")
print(f"  Action: {parsed['action']}")
print(f"  Action Input: {parsed['action_input']}")

The parser handles all three cases: a well-formed tool call, a final answer, and a malformed action input (it falls back to wrapping the raw text in a dict). This robustness is important because LLMs will occasionally deviate from the format.

---

## 7. The Scratchpad

The **scratchpad** is the full trace of the agent's reasoning. Each step adds the Thought, Action, and Observation to the message history. The LLM sees its *complete reasoning chain* on every call — this is what allows it to build on previous steps.

Here's how the messages list grows over a multi-step run:

```
Step 0 (initial):
  [system prompt]
  [user query]

Step 1 (after LLM response + tool execution):
  [system prompt]
  [user query]
  [assistant: Thought + Action + Action Input]     <-- LLM generated
  [user: Observation: <tool result>]                <-- We inject this

Step 2 (after second LLM response + tool execution):
  [system prompt]
  [user query]
  [assistant: Thought + Action + Action Input]
  [user: Observation: <tool result>]
  [assistant: Thought + Action + Action Input]     <-- LLM sees full history
  [user: Observation: <tool result>]                <-- We inject this

Step 3 (final answer):
  [system prompt]
  [user query]
  [assistant: Thought + Action + Action Input]
  [user: Observation: <tool result>]
  [assistant: Thought + Action + Action Input]
  [user: Observation: <tool result>]
  [assistant: Thought + FINAL ANSWER]              <-- Loop terminates
```

The Observation is sent as a `user` message because the LLM needs to see it as new input — it's data from the environment, not something the model generated. We also maintain a separate `scratchpad` list for our own debugging and tracing.

---

## 8. The Complete ReAct Loop

Now we bring everything together: the system prompt, tool registry, parser, and scratchpad into a single `react_agent()` function.

The loop:
1. Send messages to the LLM
2. Parse the response
3. If FINAL ANSWER, return
4. If Action, execute the tool and inject the Observation
5. If neither, nudge the LLM back on track
6. Repeat (up to `max_steps`)

In [ ]:
def react_agent(query, max_steps=10, verbose=True):
    """Run a ReAct agent loop.
    
    Args:
        query: The user's question.
        max_steps: Maximum number of reasoning steps before giving up.
        verbose: If True, print the trace as it runs.
    
    Returns:
        A dict with 'answer', 'steps', and 'scratchpad'.
    """
    system = REACT_SYSTEM_PROMPT.format(tool_descriptions=TOOL_DESCRIPTIONS)
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": query},
    ]
    
    scratchpad = []  # Track the full trace
    
    for step in range(1, max_steps + 1):
        # Call the LLM
        try:
            response = chat(messages, temperature=0)
        except RuntimeError as e:
            print(f"LLM call failed: {e}")
            return {"answer": None, "steps": step, "scratchpad": scratchpad, "error": str(e)}
        
        messages.append({"role": "assistant", "content": response})
        
        # Parse the response
        parsed = parse_react_response(response)
        
        if verbose:
            print(f"\n{'='*60}")
            print(f"Step {step}")
            print(f"{'='*60}")
            if parsed["thought"]:
                print(f"Thought: {parsed['thought']}")
        
        # Check for final answer
        if parsed["final_answer"]:
            if verbose:
                print(f"\nFINAL ANSWER: {parsed['final_answer']}")
            scratchpad.append({
                "step": step,
                "thought": parsed["thought"],
                "final_answer": parsed["final_answer"],
            })
            return {
                "answer": parsed["final_answer"],
                "steps": step,
                "scratchpad": scratchpad,
            }
        
        # Execute tool
        if parsed["action"]:
            if verbose:
                print(f"Action: {parsed['action']}")
                print(f"Action Input: {parsed['action_input']}")
            
            if parsed["action"] in TOOL_REGISTRY:
                tool_fn = TOOL_REGISTRY[parsed["action"]]
                try:
                    observation = tool_fn(**parsed["action_input"])
                except TypeError as e:
                    observation = f"Error: {e}"
            else:
                observation = (
                    f"Error: Unknown tool '{parsed['action']}'. "
                    f"Available: {list(TOOL_REGISTRY.keys())}"
                )
            
            if verbose:
                print(f"Observation: {observation}")
            
            scratchpad.append({
                "step": step,
                "thought": parsed["thought"],
                "action": parsed["action"],
                "action_input": parsed["action_input"],
                "observation": observation,
            })
            
            # Inject the observation as a user message
            messages.append({"role": "user", "content": f"Observation: {observation}"})
        else:
            # The LLM didn't follow the format — nudge it back on track
            if verbose:
                print("(No action or final answer detected — nudging the model)")
            messages.append({
                "role": "user",
                "content": "Please respond with a Thought and either an Action or FINAL ANSWER.",
            })
        
        time.sleep(1)  # Rate limiting between LLM calls
    
    return {
        "answer": None,
        "steps": max_steps,
        "scratchpad": scratchpad,
        "error": "Max steps reached",
    }

Let's start with a simple single-tool question to verify the loop works:

In [ ]:
result = react_agent("What is 25 * 48?")

The agent should have:
1. Thought about needing to calculate
2. Called the `calculate` tool
3. Received the observation (1200)
4. Produced a FINAL ANSWER

A simple question like this typically completes in 2 steps.

---

## 9. Tracing and Debugging

The scratchpad is your primary debugging tool. When an agent gives a wrong answer or gets stuck, you look at the trace to see *where the reasoning went wrong*.

Let's write a pretty-printer for the scratchpad:

In [ ]:
def print_scratchpad(result):
    """Pretty-print a ReAct agent's scratchpad."""
    print("\n" + "="*60)
    print("FULL AGENT TRACE")
    print("="*60)
    for entry in result["scratchpad"]:
        print(f"\n--- Step {entry['step']} ---")
        print(f"Thought: {entry.get('thought', 'N/A')}")
        if "action" in entry:
            print(f"Action: {entry['action']}")
            print(f"Action Input: {entry['action_input']}")
            print(f"Observation: {entry['observation']}")
        if "final_answer" in entry:
            print(f"FINAL ANSWER: {entry['final_answer']}")
    print(f"\nTotal steps: {result['steps']}")
    if result.get("error"):
        print(f"Error: {result['error']}")

In [ ]:
# Print the trace from our previous run
print_scratchpad(result)

This is how you debug agent behavior in practice. When the agent gives a wrong answer, you check the trace:

- Did the Thought correctly identify what information was needed?
- Did the Action choose the right tool?
- Did the Action Input pass the right arguments?
- Did the Observation return useful data?
- Did the next Thought correctly interpret the Observation?

Each of these is a potential failure point, and the scratchpad lets you pinpoint exactly where things went wrong.

Let's try a question that uses a different tool:

In [ ]:
result2 = react_agent("How tall is the Eiffel Tower?")
print_scratchpad(result2)

---

## Putting It Together: Multi-Step Reasoning

Now for the real test. A question that requires **multiple tools and conditional logic**:

> *"If it's above 20C in Paris, calculate a 15% tip on a $45 dinner. Otherwise, calculate a 10% tip."*

The agent needs to:
1. Call `get_weather` for Paris
2. Read the temperature from the Observation
3. Decide which tip percentage to use based on the temperature
4. Call `calculate` with the right expression
5. Combine everything into a FINAL ANSWER

This is where the Thought step really shines — you can see the agent *reasoning* about the conditional.

In [ ]:
multi_step_result = react_agent(
    "If it's above 20C in Paris, calculate a 15% tip on a $45 dinner. "
    "Otherwise, calculate a 10% tip."
)

In [ ]:
print_scratchpad(multi_step_result)

Look at the trace above. The agent should have:

1. **Step 1** — Thought about needing the weather, called `get_weather` for Paris
2. **Step 2** — Looked at the temperature in the Observation, decided which tip percentage applies, called `calculate` with the right expression
3. **Step 3** — Combined everything into a FINAL ANSWER

Compare this to what a tool-use-only agent (from NB 03) would do: it would jump straight to a tool call without explaining *why*. With ReAct, the reasoning is explicit, auditable, and — crucially — visible to the model itself, which helps it make better decisions in subsequent steps.

Let's try one more multi-step question that combines search and calculation:

In [ ]:
result3 = react_agent(
    "How tall is the Eiffel Tower in meters? If it's taller than 300 meters, "
    "calculate how many times a 1.75 meter person would need to stand on top of "
    "each other to reach that height."
)
print_scratchpad(result3)

---

## Try It Yourself

You've now built a complete ReAct agent from scratch. Here are exercises to deepen your understanding.

### Exercise 1: Add a `wikipedia_lookup` Tool

Add a new `wikipedia_lookup(topic)` tool with simulated results for a few topics (e.g., "python", "machine learning", "eiffel tower"). Wire it into the `TOOL_REGISTRY` and `TOOL_DESCRIPTIONS`, then test the agent with a question that requires it.

```python
def wikipedia_lookup(topic: str) -> str:
    """Look up a topic on Wikipedia (simulated)."""
    articles = {
        "python": "Python is a high-level programming language created by Guido van Rossum in 1991...",
        "machine learning": "Machine learning is a subset of artificial intelligence that...",
        # Add more topics
    }
    # Your implementation here
```

In [ ]:
# Exercise 1: Your code here


### Exercise 2: Count and Compare Reasoning Steps

Run the ReAct agent on several questions of varying complexity and compare how many steps each takes:

- Simple: `"What is 10 + 20?"`
- Medium: `"What is the weather in Paris?"`
- Complex: `"If it's above 25C in Paris, calculate 20% of 80. Otherwise, search for the speed of light."`

Record the step counts. Do more complex questions consistently take more steps?

In [ ]:
# Exercise 2: Your code here


### Exercise 3: ReAct vs. Plain Tool-Use

Build a simplified non-ReAct tool-use agent (similar to the pattern from NB 03) that goes directly from the question to a tool call without a Thought step. Run both agents on the same multi-step question and compare the traces.

Questions to consider:
- Does the ReAct agent make fewer errors?
- Is the ReAct trace easier to understand?
- Does the plain agent ever call the wrong tool or miss a step?

In [ ]:
# Exercise 3: Your code here


### Exercise 4: Add a Reflection Step

Modify the ReAct loop so that after each Observation, the agent produces a **Reflection** before deciding its next action. The reflection should critique its own plan:

```
Thought: I need the weather in Paris.
Action: get_weather
Action Input: {"city": "Paris"}
Observation: 22C, sunny
Reflection: The weather tool returned 22C. This is above 20C, so I should use the 15% tip rate. My plan is on track.
Thought: Now I need to calculate the 15% tip.
Action: calculate
...
```

Hint: You'll need to modify both the system prompt (to include Reflection in the format) and the agent loop (to request a Reflection after injecting each Observation).

In [ ]:
# Exercise 4: Your code here


---

## Summary

In this notebook you built a complete **ReAct agent** from scratch. Here's what came together:

| Component | What it does | From notebook |
|-----------|-------------|---------------|
| `chat()` | LLM API calls | 01 — Hello LLM |
| Agent loop | Prompt-respond-loop pattern | 02 — Basic Agent Loop |
| Tool registry + dispatch | Execute tools by name | 03 — Tool Use |
| Response parsing | Extract structured data from LLM text | 04 — Structured Output |
| **Thought step** | Explicit reasoning before each action | **NEW in this notebook** |
| **Scratchpad** | Full trace accumulation for debugging | **NEW in this notebook** |
| **FINAL ANSWER** | Clean termination signal | **NEW in this notebook** |

The key insight: adding a Thought step before each Action is a small change to the prompt, but it dramatically improves agent reliability and debuggability.

This completes **Phase 1** of the core track. You now know how to build an agent from raw Python — no frameworks, no magic.

**Next up:** Phase 2 of the core track will introduce agent frameworks and more advanced patterns. Stay tuned!